# Step 8 : 数値安定化について

## 1: $Q_{uu,n}$の正則性と正定値化

[Step5 backward pass](/notes/step5.ipynb)の入力修正則の導出において、Q関数の二次近似式の$Q_n$を$\delta u_n$で偏微分して、ゼロとなる停留点を最小点の候補であると導出した。

これが最小点となるための十分条件は、$Q_n$の$\delta u_n$によるHessianが正定値であることであった。

$$
\frac{\partial^2 Q_n}{\partial \ \delta u_n^2} = Q_{uu,n} \succ 0
$$

$Q_{uu,n}$は以下のステージコストの$u$によるHessianと、価値関数の状態$X$によるHessianで構成される。

$$
Q_{uu,n} = \ell_{uu,n} + B_n^{T} V_{XX,n+1} B_n
$$

$\ell_{uu,n} \succ 0, V_{XX,n+1} \succeq 0$ であれば、$Q_{uu,n} \succ 0$ となるが、$V_{XX,n+1},\ell_{uu,n}$は非線形な非凸な問題では負の曲率を持つ場合があり、$Q_{uu,n}$が必ずしも正定値であることは通常保証されない。

そこで次のように、$Q_{uu,n}$が正定値となるように、十分な大きさの正則化係数$\mu \ge 0$を選び、以下のように$\tilde{Q}_{uu,n}$を構成する。

$$
\tilde{Q}_{uu,n} = Q_{uu,n} + \mu I
$$

$\tilde{Q}_{uu,n} \succ 0$ であれば、次の入力修正則$\delta u_n$は、正則化後の局所二次Q関数を$\delta X_n$を固定して、入力の摂動$\delta u_n$について一意に最小化する。

$$
\delta u_n = k_n + K_n \delta X_n
$$

$$
k_n = -\tilde{Q}_{uu,n}^{-1} Q_{u,n}, \quad K_n = -\tilde{Q}_{uu,n}^{-1} Q_{uX,n}
$$

### 入力ペナルティ

$Q_n$を$\delta u_n$に関する項でまとめると次のようになる。$C$は$\delta u_n$に依存しない項である。

$$
Q_n(\delta u_n) = (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ Q_{uu,n} \ \delta u_n + C 
$$

これに入力ペナルティを加えると、次のようになり、

$$
\begin{aligned}
\tilde{Q}_n &= Q_n(\delta u_n) + \frac{\mu}{2} \delta u_n^T \delta u_n \\
&= (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ \left(Q_{uu,n} +  \mu I \right)\ \delta u_n + C 
\end{aligned}
$$

この$\delta u_n$についてのHessianが$\tilde{Q}_{uu,n}$である。

$$
\tilde{Q}_{uu,n} = Q_{uu,n} +  \mu I
$$ 

つまり、$\mu$は$\delta u_n$に対する重みであるため、大きくすると、$\delta u_n$を小さくする効果がある。

### 正則化係数$\mu$の調整方法

前節の通り、$\mu$を大きくすると入力修正則が小さくなるので、軌道の改善が遅くなる。そこで、最初から大きな値ではなく、次のように調整を行う。

$\tilde{Q}_{uu,n}$が正定値行列であるかはCholesky分解の結果より判別することが出来る。

backward passではすでにCholesky分解を用いた$k_n, K_n$の計算と、Cholesky分解を行う$Q_{uu,n}$の対称化処理を行っている。

これに正則化係数$\mu$の調整を加える。

backward passに引数として$\mu$を与える。そしてCholesky分解では対称化した$\tilde{Q}_{uu,n}(\mu)$の分解を行う。もし分解に失敗したら、$\tilde{Q}_{uu,n}(\mu)$ は正定値対称行列ではないとなるため、$\mu$を調整してbackward pass を再計算する。

$\mu$の初期値$\mu_{\text{init}} > 0$とし、backward pass を再計算する場合は、$\mu \leftarrow \max(\mu_{\text{init}}, 10 \mu)$ とする。

また、backward pass ではcholesky分解の結果が示すのは$\tilde{Q}_{uu,n}(\mu)$が正定値行列であることであり、実際にコストが下がるかは forward pass で確認する必要がある。

そこで、forward passのline searchでもコストが下がらない場合、$\mu$を増やしてbackward passを再計算する。コストが下がれば、次の反復では$\mu = \mu_{\text{init}}$として計算を行う。<br>
または前回の反復で用いた用いた$\mu$を用いて、$\mu \leftarrow \max(\mu_{\text{init}}, \mu/10)$のように徐々に減らしながら調整する方法もある。


## 2 : 予測コスト減少量と実際の減少量の比較


### 1時刻での予測コスト変化

局所二次Q関数は次の式である。

$$
\begin{aligned}
Q_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n) \approx & \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

この式で、$\delta X_n=0$として入力を$\delta u_n = \alpha k_n$だけ変化させると、以下となる。

$$
Q_n(\bar{X}_n, \bar{u}_n+ \alpha k_n) - Q_n(\bar{X}_n, \bar{u}_n) \approx \alpha Q_{u,n}^T  k_n + \frac{1}{2} \alpha^2 k_n^T Q_{uu,n} k_n
$$ 

上式の右辺はコストの変化量であるため、負なら減少を意味する。

以下のように$d_{1,n}, d_{2,n}$を定義すれば、

$$
d_{1,n} =Q_{u,n}^T  k_n , \quad d_{2,n} = \frac{1}{2} k_n^T Q_{uu,n} k_n
$$

次のようにコストの変化分を記述できる。

$$
\Delta Q_n(\alpha) = \alpha d_{1,n} + \alpha^2 d_{2,n}
$$

正則化を行っていない$Q_{uu,n}$では、$Q_{uu,n} k_n = - Q_{u,n}$であるため、コストの変化分は以下となる。

$$
\begin{aligned}
\Delta Q_n(\alpha) &= - \alpha (Q_{uu,n} k_n)^T  k_n + \frac{1}{2} \alpha^2 k_n^T Q_{uu,n} k_n \\
&= - \alpha k_n^T Q_{uu,n}^T  k_n + \frac{1}{2} \alpha^2 k_n^T Q_{uu,n}  k_n \\
&= - \alpha (1 - \frac{\alpha}{2}) k_n^T Q_{uu,n} k_n 
\end{aligned}
$$

$Q_{uu,n} \succ 0, k_n \neq 0, 0 < \alpha \le 1$ の条件では$\Delta Q_n(\alpha)< 0$となる。 




### ホライゾン全体の予測コスト減少量

以下では、以下の条件で予測コストの減少量を考える。

- 正則化なしの$Q_{nn,n} \succ 0$ 
- 初期状態の摂動 $\delta X_0 = 0$
- forward passでは線形化モデルで生成した摂動軌道を計算
- 予測コストは総コストの近似式で計算

Q関数の二次近似式$\hat{Q}_n$を以下で定義する。

$$
\begin{aligned}
\hat{Q}_n(\delta X_n, \delta u_n) \coloneqq & \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

上式は以下の式を用いて、

$$
C_n(\delta X_n) = \bar{Q}_n + Q_{X,n}^T \delta X_n+ \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n , \quad b_n = Q_{u,n} + Q_{uX,n} \delta X_n
$$

以下のように書ける。

$$
\hat{Q}_n(\delta X_n, \delta u_n) = C_n(\delta X_n) + b_n^T \delta u_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
$$

$\delta X_n$を固定し、$\hat{Q}_n$を最小化する入力を $\delta u_n^\star$ と書くと以下である。

$$
\delta u_n^\star = -Q_{uu,n}^{-1} b_n = k_n + K_n \delta X_n
$$

よって、以下となる。

$$
b_n = - Q_{uu,n} \delta u_n^\star
$$

これより、

$$
\begin{aligned}
\hat{Q}_n(\delta X_n, \delta u_n) &= C_n(\delta X_n) - (\delta u_n^\star)^T Q_{uu,n} \delta u_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n \\
&= C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T Q_{uu,n} \delta u_n^\star \\
& \quad + \frac{1}{2} (\delta u_n - \delta u_n^\star) ^T Q_{uu,n} (\delta u_n - \delta u_n^\star) \\
\end{aligned}
$$

$$
\hat{V}_n(\delta X_n) \coloneqq \hat{Q}_n(\delta X_n, \delta u_n^\star) = C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T Q_{uu,n} \delta u_n^\star
$$

と置くと、以下となる。

$$
\begin{aligned}
\hat{Q}_n(\delta X_n, \delta u_n) =\hat{V}_n(\delta X_n) + \frac{1}{2} (\delta u_n - \delta u_n^\star) ^T Q_{uu,n} (\delta u_n - \delta u_n^\star) 
\end{aligned}
$$

ここに、forward passの修正項を代入すると、

$$
\delta u_n = \alpha k_n + K_n \delta X_n
$$

$$
\begin{aligned}
\hat{Q}_n(\delta X_n, \delta u_n) & = \hat{V}_n(\delta X_n) \\
& + \frac{1}{2} (\alpha k_n + \cancel{K_n \delta X_n} -  k_n - \cancel{K_n \delta X_n}) ^T Q_{uu,n} (\alpha k_n + \cancel{K_n \delta X_n} -  k_n - \cancel{K_n \delta X_n}) \\
&= \hat{V}_n(\delta X_n) + \frac{1}{2}(- (1- \alpha) k_n)^T Q_{uu,n} (- (1- \alpha) k_n) \\
&= \hat{V}_n(\delta X_n) + \frac{1}{2} (1 - \alpha)^2 k_n^T Q_{uu,n} k_n \\
\end{aligned}
$$

元の非線形なQ関数は次の定義である。

$$
\begin{aligned}
Q_n(X_n, u_n) = \ell_n(X_n, u_n) + V_{n+1}(X_{n+1}) , \quad X_{n+1} &= f(X_n, u_n)
\end{aligned}
$$


これを局所近似したQ関数は以下となる。$\hat{\ell}_n$はステージコストの二次近似である。

$$
\hat{Q}_n(\delta X_n , \delta u_n) = \hat{\ell}_n (\delta X_n, \delta u_n) + \hat{V}_{n+1}(\delta X_{n+1}) , \quad \delta X_{n+1} = A_n \delta X_n + B_n \delta u_n
$$

上式は先ほどまとめた$\hat{Q}_n(\delta X_n , \delta u_n)$と同じであるため、ステージコストの二次近似は次のようになる。

$$
\begin{array}{c}
\hat{V}_n(\delta X_n) + \frac{1}{2} (1 - \alpha)^2 k_n^T Q_{uu,n} k_n = \hat{\ell}_n (\delta X_n, \delta u_n) + \hat{V}_{n+1}(\delta X_{n+1}) \\
\downarrow \\
\hat{\ell}_n (\delta X_n, \delta u_n) = \hat{V}_n(\delta X_n) - \hat{V}_{n+1}(\delta X_{n+1})+ \frac{1}{2} (1 - \alpha)^2 k_n^T Q_{uu,n} k_n 
\end{array}
$$

予測コストは、総コストの近似式を用いて次のように計算する。

$$
\hat{J} = \hat{\phi}(\delta X_N) + \sum_{n=0}^{N-1} \hat{\ell}_n
$$

$\hat{\ell}_n$ を$n=0, ... , N-1$で加算していく。価値関数の部分だけを見ると、

$$
\begin{aligned}
 &[\hat{V}_0(\delta X_0) - \hat{V}_1(\delta X_1)]\\
+&[\hat{V}_1(\delta X_1) - \hat{V}_2(\delta X_2)]\\
+&\cdots\\
+&[\hat{V}_{N-1}(\delta X_{N-1}) - \hat{V}_N(\delta X_N)]\\
\end{aligned}
$$

であるため、途中の項が打ち消しあい、次のようになる。

$$
\sum_{n=0}^{N-1} \hat{\ell}_n = \hat{V}_0(\delta X_0) - \hat{V}_N(\delta X_N) + \frac{1}{2} (1 - \alpha)^2 \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n
$$ 

$\hat{V}_N(\delta X_N) = \hat{\phi}(\delta X_N)$ であるため、予測コストは以下となる。

$$
\hat{J}(\alpha) = \hat{V}_0(\delta X_0)  + \frac{1}{2} (1 - \alpha)^2 \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n
$$

初期摂動の状態は$\delta X_0 = 0$とし、線形化モデルでforward passを計算するので、$\alpha=0$とすると、以下のように状態摂動、入力摂動ともにゼロとなる。

$$
\begin{aligned}
 (\delta X_0 = X^{\text{new}} - \bar{X}_0 = 0) &\rightarrow (\delta u_0 = K_0 \delta X_0 = 0) \\
\rightarrow  (\delta X_1 = A_0 \delta X_0 + B_0 \delta u_0 = 0 ) &\rightarrow (\delta u_1 = K_1 \delta X_1 = 0) \\
\vdots \\
(\delta X_{n+1} = A_n \delta X_n + B_n \delta u_n = 0 ) &\rightarrow (\delta u_{n+1} = K_{n+1} \delta X_{n+1} = 0)
\end{aligned}
$$

よって、基準軌道による予測コスト$\hat{J}(0)$となる。

$$
\hat{J}(0) = \hat{V}_0(\delta X_0)  + \frac{1}{2} \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n
$$

よって、$\alpha k_n$ による、予測コスト変化$\Delta \hat{J}$は$\hat{J}(\alpha) - \hat{J}(0)$となるため、

$$
\Delta \hat{J} = - \alpha (1 - \frac{\alpha}{2}) \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n 
$$

